# `catcif-storage/pipeline.py` — output walkthrough

After running `./pipeline.py` (RFD3 → MPNN → AF3 → RMSD with `auto_store_format="catcif"`, `auto_store_cleanup=True`), this notebook walks through the artifacts it left behind and shows how to:

1. Inspect the per-chunk catcif archives.
2. Load structures directly from catcif tags via `parse_to_atom_array`.
3. Consolidate per-chunk archives into one-per-step via `handler.store("all", dir, format="catcif")`.
4. Materialize structures back to loose `.cif` files with `handler.materialize(...)`.
5. Visualize a loaded structure.

Prereq: the pipeline must have completed successfully and left the rundir in place.

In [ ]:
from pathlib import Path


from chaperone.data import PipelineDataHandler
from chaperone.data.catcif import list_catcif_entries

RUNDIR = Path(".").resolve() / "rundir"
assert RUNDIR.exists(), f"Run ./pipeline.py first; RUNDIR not found at {RUNDIR}"
print("rundir:", RUNDIR)

## 1. What's on disk?

With `auto_store_cleanup=True` the per-chunk sandboxes get removed. What's left in each step dir: the per-chunk catcif archives + the step-level `output.parquet` (+ a debug `output.csv`).

In [ ]:
for step in ("rfd3", "mpnn", "af3"):
    step_dir = RUNDIR / step
    if not step_dir.exists():
        continue
    files = sorted(p for p in step_dir.iterdir() if p.is_file())
    print(f"\n=== {step} ===")
    for p in files:
        print(f"  {p.name:<40} {p.stat().st_size:>8} B")

Each `<step>.chunk_N.catcif` is one SLURM chunk's worth of structures. `list_catcif_entries` shows the entry ids stored inside:

In [ ]:
for p in sorted((RUNDIR / "af3").glob("*.chunk_*.catcif")):
    entries = list_catcif_entries(p)
    print(f"{p.name}: {len(entries)} entries → {entries[:3]}...")

## 2. Load the final handler and see where paths point

The pipeline's final parquet sits at `rundir/rmsd/output.parquet` (last step). Structure paths for each ancestor step are catcif tags — they resolve transparently when loaded.

In [ ]:
handler = PipelineDataHandler.from_file(RUNDIR / "rmsd" / "output.parquet")
print(f"rows = {len(handler)}, steps = {handler.names}")
unpacked = handler.unpack(to_pandas=False)
unpacked.select(["rfd3.path", "mpnn.path", "af3.path"]).head(3)

In [ ]:
# Pick a representative RMSD column to confirm metrics survived
rmsd_cols = [c for c in unpacked.columns if c.startswith("rmsd.")]
unpacked.select(rmsd_cols).describe()

## 3. Load a structure directly from its catcif tag

`parse_to_atom_array` dereferences catcif tags via `catcif_tools` — no manual extraction needed.

In [ ]:
from chaperone.io import parse_to_atom_array

sample_tag = unpacked["af3.path"][0]
print("tag:", sample_tag)
atoms = parse_to_atom_array(sample_tag)
print(f"AtomArray: {len(atoms)} atoms, chains = {sorted(set(atoms.chain_id))}")

## 4. Consolidate per-chunk archives into one-per-step

`handler.store("all", dir, format="catcif")` walks every step with structures and writes a single `<step>.catcif` per step under `dir`. Per-chunk files remain in place — this is a *copy*, not a move.

In [ ]:
PACKAGED = RUNDIR / "packaged"
handler.store("all", PACKAGED, format="catcif")

for p in sorted(PACKAGED.iterdir()):
    if p.suffix == ".catcif":
        print(
            f"{p.name:<20} {len(list_catcif_entries(p))} entries  ({p.stat().st_size} B)"
        )

### `which=` variants at a glance

- `store("last", dir, ...)` — only the most recent step (here: `rmsd` has no structures since it's a metric-only block, so the last step with structures is `af3`).
- `store("pending", dir, ...)` — only steps that don't yet have `store_path` set. Useful if you've already archived some steps and want to incrementally handle the rest.

In [ ]:
LAST_ONLY = RUNDIR / "packaged_last_only"
# Reload a fresh handler so store_path annotations don't shift what 'last' resolves to
h2 = PipelineDataHandler.from_file(RUNDIR / "rmsd" / "output.parquet")
h2.store("last", LAST_ONLY, format="catcif")
print("last-only contents:")
for p in sorted(LAST_ONLY.iterdir()):
    print(" ", p.name)

## 5. Materialize to loose `.cif` files

When you need real files (e.g. to hand off to a wet-lab tool that can't speak catcif), `handler.materialize(dir)` extracts every stored entry to an individual `.cif` file. Pass `expand_catcif=False` to leave catcif tags untouched (useful if the next consumer is also catcif-aware).

In [ ]:
# Package archives first (store="all") so materialize has a single store_path to read from
h3 = PipelineDataHandler.from_file(RUNDIR / "rmsd" / "output.parquet")
h3.store("all", RUNDIR / "packaged_for_materialize", format="catcif")

MAT = RUNDIR / "materialized"
h3.materialize(MAT, expand_catcif=True)

for step_dir in sorted(MAT.iterdir()):
    if step_dir.is_dir():
        cifs = sorted(step_dir.glob("*.cif"))
        print(f"{step_dir.name}: {len(cifs)} files")

## 6. Visualize a design

Any catcif tag (or a materialized `.cif`) loads into a biotite `AtomArray` we can inspect or send to a viewer. Below we dump a small summary; hook into PyMOL via the `/visualize` skill if you want a 3D view.

In [ ]:
import numpy as np
from chaperone.io import parse_to_atom_array

sample_tag = unpacked["af3.path"][0]
atoms = parse_to_atom_array(sample_tag)

# Summary
chains = sorted(set(atoms.chain_id))
n_res = len({(c, r) for c, r in zip(atoms.chain_id, atoms.res_id)})
ca = atoms[atoms.atom_name == "CA"]
radius_of_gyration = float(np.sqrt(((ca.coord - ca.coord.mean(0)) ** 2).sum(-1).mean()))
print(f"design    : {sample_tag.rsplit(':', 1)[1]}")
print(f"atoms     : {len(atoms)}")
print(f"chains    : {chains}")
print(f"residues  : {n_res}")
print(f"Rg (CA)   : {radius_of_gyration:.2f} Å")

In [ ]:
# Example: open in PyMOL via XML-RPC (uncomment if a pymol server is running)
# import xmlrpc.client
# s = xmlrpc.client.ServerProxy('http://localhost:9123')
# cif_path = MAT / 'af3' / f"{sample_tag.rsplit(':', 1)[1]}.cif"
# s.load(str(cif_path))
# s.show_as('cartoon')
# s.color('teal')
print("use /visualize skill or pymol RPC to render interactively")

## Summary

- Per-chunk catcif archives live at `rundir/<step>/<step>.chunk_N.catcif`. Paths in the handler are self-describing (`archive.catcif:entry_id`).
- `handler.store(step_or_sentinel, dir, format="catcif", cleanup=...)` is the on-demand knob: one step, `"last"`, `"all"`, or `"pending"`.
- `handler.materialize(dir, expand_catcif=True)` turns archives back into individual `.cif` files — drop `expand_catcif=False` to keep catcif tags.
- Any downstream tool that reads via `parse_to_atom_array` / `CompiledBlockFunction.stage_file` / `LoadedStructure` is catcif-native for free.